# 06 · Contraction with einsum / Contracción con einsum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/06-contraction-with-einsum.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#db2777,rgba(219,39,119,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#db2777">PART IV · EXERCISE · 15 MIN</span>

## Practise today / Practica hoy

Contract colour with einsum, name surviving axes, and verify a weighted pixel sum.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Contraer el color con einsum, nombrar los ejes restantes y verificar una suma ponderada de píxeles.</div></div>

## Explore later / Explora después

Express matrix operations with einsum and compare digit similarity measures.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Expresar operaciones matriciales con einsum y comparar medidas de similitud de dígitos.</div></div>

Follow the core block immediately below. / Sigue el bloque esencial de abajo.

<!-- CORE-PATH -->
## Core path / Ruta esencial

Read and run this block from top to bottom: **recall → example → attempt → feedback → checkpoint**. Preparation and feedback definitions appear where needed. Try before opening a folded solution. Stop at **Core complete**; everything after it is **Explore later**.

🇪🇸 Lee y ejecuta este bloque de arriba abajo: **recuerda → ejemplo → intento → retroalimentación → comprobación**. La preparación y las funciones de comprobación aparecen donde se necesitan. Inténtalo antes de abrir una solución plegada. Detente en **Fin de la ruta esencial**; después empieza **Explora después**.

### Recall / Recuerda

Recall notebook 05: what does each axis of a sampled video (T,H,W,C) count? Which should remain if you remove colour?

🇪🇸 Recuerda el cuaderno 05: ¿qué cuenta cada eje de un video muestreado (T,H,W,C)? ¿Cuáles deben permanecer al quitar el color?

## Setup / Preparación

We use two real datasets already included in standard Python libraries:

- a real colour microscopy image from `skimage.data`;
- the 1,797-image handwritten-digits dataset from `sklearn`.

The grayscale vector `w` is **not observed data**. It is a transformation rule that tells us how strongly red, green, and blue contribute to one grayscale value.

> 🇪🇸 Usaremos dos conjuntos de datos reales incluidos en librerías estándar:
>
> - una imagen real de microscopía a color de `skimage.data`;
> - el conjunto de 1.797 imágenes reales de dígitos manuscritos de `sklearn`.
>
> El vector de pesos `w` **no es un conjunto de datos observado**. Es una regla de transformación que indica cuánto contribuyen rojo, verde y azul a un valor en escala de grises.

### Core prep 1/1 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from sklearn.datasets import load_digits
from skimage import data

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# ---------------------------------------------------------------------------
# Real colour images
# ---------------------------------------------------------------------------
photo = data.immunohistochemistry().astype(float)          # (512, 512, 3)
batch = np.stack([
    photo,
    data.astronaut().astype(float),
])                                                         # (2, 512, 512, 3)

# RGB -> grayscale transformation weights.
w = np.array([0.2125, 0.7154, 0.0721])

# ---------------------------------------------------------------------------
# Real handwritten digits
# ---------------------------------------------------------------------------
digits = load_digits()
digit_images = digits.images.astype(float)                 # (1797, 8, 8)

# Tiny matrices for Exercise 2 come from real digit pixels.
A = digit_images[0, 2:4, 2:4]
B = digit_images[1, 2:4, 2:4]

print("Photo / Foto:", photo.shape)
print("Image batch / Lote de imágenes:", batch.shape)
print("Digits / Dígitos:", digit_images.shape)
print("Labels / Etiquetas:", digits.target.shape)
print()
print("A comes from digit label / A proviene del dígito:", int(digits.target[0]))
print(A)
print()
print("B comes from digit label / B proviene del dígito:", int(digits.target[1]))
print(B)
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## How to read `einsum` / Cómo leer `einsum`

Think of the letters as **axis names**, not mysterious algebra.

| Expression / Expresión | What disappears? / ¿Qué desaparece? | Output meaning / Significado de salida |
|---|---|---|
| `hwc,c->hw` | `c` | one value per `(h,w)` pixel / un valor por píxel `(h,w)` |
| `ii->` | `i` | one scalar / un escalar |
| `ij->ji` | nothing / nada | same values, axes reversed / mismos valores, ejes invertidos |
| `ik,kj->ij` | `k` | matrix product / producto matricial |
| `id,jd->ij` | `d` | one score for each pair `(i,j)` / una puntuación por cada par `(i,j)` |

### The sentence to remember / La frase para recordar

> **Before the arrow = available indices. After the arrow = indices you want to keep. Missing indices are summed.**

$$
\underbrace{\mathrm{ijk,k}}_{\text{available}}
 \rightarrow 
\underbrace{\mathrm{ij}}_{\text{kept}}
\qquad\Longleftrightarrow\qquad
C_{ij} = \sum_{k} A_{ijk} w_{k}
$$

Read it as: an index on the left but not on the right is summed away; an index
on the right but on neither side of the comma is a new axis. That one rule is
the whole of `einsum`.

> 🇪🇸
>
> **Antes de la flecha = índices disponibles. Después de la flecha = índices que quieres conservar. Los índices que desaparecen se suman.**

### The index that disappears / El índice que desaparece

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#db2777,rgba(219,39,119,0))"></div>

One contraction, at the size of its own arithmetic, with a colour per index: <code>i</code> blue, <code>j</code> orange, <code>k</code> green. A fibre of five numbers along <code>k</code> meets five weights that share that <code>k</code>, the five products are added into one output cell, and the last frame has no <code>k</code> arrow left to draw.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-06-contract.gif" alt="An animation of the contraction einsum ijk,k to ij, with each index drawn in its own colour. The three axes of a 3 by 4 by 5 tensor are labelled i, j and k. A fibre of five cells along k is highlighted in green beside a green vector of five weights, the two are multiplied elementwise and the five products are added to 15, and 15 appears in the corner of a 3 by 4 output that carries only an i arrow and a j arrow." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una contracción, al tamaño de su propia aritmética, con un color por índice: <code>i</code> azul, <code>j</code> naranja, <code>k</code> verde. Una fibra de cinco números a lo largo de <code>k</code> se encuentra con cinco pesos que comparten esa <code>k</code>, los cinco productos se suman en una sola celda de salida, y en el último fotograma ya no queda ninguna flecha <code>k</code> que dibujar.</div>

### One pixel by hand / Un píxel a mano

A pixel `[10, 20, 30]` and weights `[0.2, 0.5, 0.3]` give
`10×0.2 + 20×0.5 + 30×0.3 = 21`. Three colour values become one value.
For two pixels `[[10, 20, 30], [0, 10, 20]]`, the same weights give `[21, 11]`:
`pc,c->p` keeps the pixel index and sums the colour index.

🇪🇸 Un píxel `[10, 20, 30]` y pesos `[0.2, 0.5, 0.3]` dan
`10×0.2 + 20×0.5 + 30×0.3 = 21`: tres valores de color se convierten en uno.
Para dos píxeles `[[10, 20, 30], [0, 10, 20]]`, los mismos pesos dan `[21, 11]`:
`pc,c->p` conserva el índice de píxel y suma el de color.


## Exercise 1 — contract the colour axis / Ejercicio 1 — contrae el eje de color

`photo` is a real microscopy image:

`photo.shape = (512, 512, 3)`

Read that as:

`(H, W, C) = height × width × colour`

The contraction is:

`hwc,c->hw`

### Predict first / Predice primero

1. Which index disappears?
2. Which indices survive?
3. Why must the result have shape `(512,512)`?

> 🇪🇸 `photo` es una imagen real de microscopía con forma `(512,512,3)`.
>
> En `hwc,c->hw`, predice:
>
> 1. ¿qué índice desaparece?
> 2. ¿qué índices permanecen?
> 3. ¿por qué el resultado debe tener forma `(512,512)`?

### Optional hints / Pistas opcionales

Try first; open one hint at a time. / Inténtalo primero; abre una pista a la vez.

<details>
<summary>Hint 1 / Pista 1</summary>

For one pixel, multiply its three channel values by the corresponding weights. Which index disappears when you add those products?

🇪🇸 Para un píxel, multiplica sus tres canales por los pesos correspondientes. ¿Qué índice desaparece al sumar esos productos?

</details>

<details>
<summary>Hint 2 / Pista 2</summary>

List every axis that must survive to the right of ->. In the batch expression, keep the image index as well as both spatial indices.

🇪🇸 Enumera a la derecha de -> cada eje que debe permanecer. En el lote, conserva el índice de imagen y ambos índices espaciales.

</details>



In [ ]:
# Feedback helper / Función de comprobación — run before your attempt / ejecuta antes del intento
import numpy as np

def check_core_answer(photo, batch, w, gray, gray_batch):
    """Check colour contraction independently of einsum / Comprueba la contracción sin einsum."""
    photo, batch, w, gray, gray_batch = map(np.asarray, (photo, batch, w, gray, gray_batch))
    assert photo.ndim == 3 and batch.ndim == 4, "Inputs must be HWC and NHWC / Entradas HWC y NHWC."
    assert w.shape == (photo.shape[-1],) and batch.shape[-1] == len(w), "One weight per colour / Un peso por color."
    assert gray.shape == photo.shape[:-1], "Keep both spatial axes / Conserva ambos ejes espaciales."
    assert gray_batch.shape == batch.shape[:-1], "Keep batch and spatial axes / Conserva lote y ejes espaciales."
    assert np.isfinite(gray).all() and np.isfinite(gray_batch).all(), "Output must be finite / La salida debe ser finita."
    assert np.allclose(gray, (photo * w).sum(axis=-1)), "Check weights and contracted axis / Revisa pesos y eje contraído."
    assert np.allclose(gray_batch, (batch * w).sum(axis=-1)), "Check every image, retaining batch order / Revisa cada imagen conservando el orden del lote."
    return "Checks passed; explain which index disappears / Comprobaciones superadas; explica qué índice desaparece."


### Core activity · Actividad esencial

**Predict → Run → Explain → Check**

1. **Predict.** Which index disappears in nhwc,c->nhw?
2. **Run.** Complete Exercise 1.
3. **Explain.** Explain why the batch axis survives.
4. **Check.** Run `check_core_answer(photo, batch, w, gray, gray_batch)` on your own results before opening the solution. Compare your batch result with (batch * w).sum(axis=-1) using np.allclose.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

1. **Predice.** ¿Qué índice desaparece en nhwc,c->nhw?
2. **Ejecuta.** Completa el Ejercicio 1.
3. **Explica.** Explica por qué se conserva el eje de lote.
4. **Comprueba.** Ejecuta `check_core_answer(photo, batch, w, gray, gray_batch)` con tus resultados antes de abrir la solución. Compara tu resultado con (batch * w).sum(axis=-1) usando np.allclose.

</details>

Prediction / Predicción: ___  
Evidence / Evidencia: ___  
Revised explanation / Explicación revisada: ___

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# Use one einsum to convert `photo` to grayscale:
#     'hwc,c->hw'
# Expected shape: (512, 512)
#
# ES:
# Usa un solo einsum para convertir `photo` a gris:
#     'hwc,c->hw'
# Forma esperada: (512, 512)
#
# TODO 2 / TAREA 2
#
# EN:
# Do the same for the entire real image batch in ONE einsum:
#     Write the index expression / Escribe la expresión de índices
# Expected shape: (2, 512, 512)
#
# ES:
# Haz lo mismo para todo el lote real en UN solo einsum:
#     Write the index expression / Escribe la expresión de índices
# Forma esperada: (2, 512, 512)
#
# Explain / Explica:
# - which index is contracted / qué índice se contrae
# - which indices survive / qué índices permanecen
# Check your own results before opening the solution / Comprueba tus resultados antes de abrir la solución:
# check_core_answer(photo, batch, w, gray, gray_batch)


In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

gray = np.einsum("hwc,c->hw", photo, w)
gray_batch = np.einsum("nhwc,c->nhw", batch, w)

print("Single image / Imagen individual:")
print(photo.shape, "->", gray.shape)
print()

print("Batch / Lote:")
print(batch.shape, "->", gray_batch.shape)
print()

print("EN: c disappears, so colour is multiplied by weights and summed.")
print("ES: c desaparece, por lo que color se multiplica por pesos y se suma.")
print("EN: h and w survive; n also survives for the batch.")
print("ES: h y w permanecen; n también permanece en el lote.")

fig, axes = plt.subplots(1, 2, figsize=(8.5, 4))

axes[0].imshow(photo / 255.0)
axes[0].set_title("Real RGB input / Entrada RGB real")

axes[1].imshow(gray, cmap="gray")
axes[1].set_title("hwc,c->hw\nc disappears / c desaparece")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()
print(check_core_answer(photo, batch, w, gray, gray_batch))


<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

At every pixel `(h,w)` there are three numbers:

`[R, G, B]`

and three weights:

`[wR, wG, wB]`

`einsum('hwc,c->hw', photo, w)` calculates:

`R×wR + G×wG + B×wB`

$$
G_{hw} = \sum_{c \in \{R, G, B\}} P_{hwc} w_{c}
$$


for every pixel.

The colour index `c` is no longer needed after the sum, so it disappears.

> 🇪🇸 En cada píxel `(h,w)` existen tres números `[R,G,B]` y tres pesos.
>
> `einsum('hwc,c->hw', photo, w)` calcula una suma ponderada para cada píxel.
>
> Después de sumar sobre color, el índice `c` ya no es necesario y desaparece.

</details>

### Checkpoint / Comprobación

**Independent checkpoint · 1 minute within Explain/Check.** After checking your code, close the hints and solution. Answer alone before comparing with a partner. A new tensor `S` has shape `(2, 3, 4)` = (batch, time, feature), with weights `w` of shape `(4,)`. Write an einsum that keeps batch and time. State its output shape and contracted index. What would `ntf,f->n` do differently?

**Comprobación independiente · 1 minuto dentro de Explica/Comprueba.** Tras comprobar tu código, cierra las pistas y la solución. Responde individualmente antes de comparar con otra persona. Un nuevo tensor `S` tiene forma `(2, 3, 4)` = (lote, tiempo, característica), con pesos `w` de forma `(4,)`. Escribe un einsum que conserve lote y tiempo. Indica su forma de salida y el índice contraído. ¿Qué haría distinto `ntf,f->n`?

<details>
<summary>Checkpoint answer — attempt first / Respuesta — inténtalo primero</summary>

`np.einsum("ntf,f->nt", S, w)` returns `(2, 3)` and sums `f`. `ntf,f->n` sums both `f` and `t`, returning `(2,)` and losing the separate time positions.

🇪🇸 `np.einsum("ntf,f->nt", S, w)` devuelve `(2, 3)` y suma `f`. `ntf,f->n` suma tanto `f` como `t`, devuelve `(2,)` y pierde las posiciones temporales separadas.

</details>


## Core complete / Fin de la ruta esencial

Keep your prediction, evidence, and explanation. Follow the facilitator’s quiz and break schedule before continuing.

🇪🇸 Guarda tu predicción, evidencia y explicación. Sigue las pausas y quizzes del facilitador antes de continuar.

[Next: Notebook 07 / Siguiente: cuaderno 07](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/07-inverses-and-pseudoinverse.ipynb).

## Explore later / Explora después

Optional reference, exercises, and explorers. These are outside this section’s live core. Continue in order when studying them; some reuse earlier setup.

🇪🇸 Material de consulta, ejercicios y exploradores opcionales. Quedan fuera de la ruta esencial en vivo. Continúa en orden al estudiarlos; algunos reutilizan la preparación anterior.

## One rule at three scales / Una regla a tres escalas

Contraction is not a matrix-algebra trick. Reducing three RGB measurements to one grayscale value, multiplying two matrices, comparing one image vector against thousands: these are the same operation with different letters. Multiply matching positions, add them, and the summed axis disappears.

> 🇪🇸 Una contracción no es un truco de álgebra matricial. Convertir tres valores RGB en un gris, multiplicar dos matrices, comparar un vector de imagen contra miles: son la misma operación con otras letras. Multiplicar posiciones correspondientes, sumarlas, y el eje sumado desaparece.


### The index that appears / El índice que aparece

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#db2777,rgba(219,39,119,0))"></div>

The companion to contraction, in the same colours. An index written after the arrow but on neither side of the comma is a new axis, so <code>'i,j-&gt;ij'</code> is the outer product: blue <code>i</code> against orange <code>j</code> makes a grid carrying both. The last frame sums <code>j</code> away again — the arrow decides, in both directions.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-06-outer.gif" alt="An animation showing a blue 4-element vector and an orange 5-element vector, then their outer product as a 4 by 5 grid with an i arrow and a j arrow, then one entry traced back to the two values that made it, then the grid contracted back to a blue 4-element vector." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El complemento de la contracción, en los mismos colores. Un índice escrito después de la flecha pero en ningún lado de la coma es un eje nuevo, así que <code>'i,j-&gt;ij'</code> es el producto exterior: la <code>i</code> azul contra la <code>j</code> naranja crea una cuadrícula que lleva ambas. El último fotograma vuelve a sumar la <code>j</code>: la flecha decide, en ambas direcciones.</div>

### The einsum you already write as `@` / El einsum que ya escribes como `@`

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#db2777,rgba(219,39,119,0))"></div>

Matrix multiplication is a contraction, and nothing else. A row meets a column, the three products are added, and `k` — the index that sits on both operands and never after the arrow — is gone. The last frame is the one worth carrying away: put `b` on both operands *and* after the arrow and it is carried rather than summed, which is how a library multiplies a whole batch of matrices without a loop.

$$
\mathrm{out}_{ij} = \sum_{k} A_{ik} B_{kj}
\qquad\qquad
\mathrm{out}_{bij} = \sum_{k} X_{bik} Y_{bkj}
$$

Read it as: the same sum twice, with $b$ carried along untouched on both sides
of it. An index written on every operand *and* after the arrow is a spectator —
the contraction simply happens once for each value it takes.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-06-matmul.gif" alt="An animation of a two by three matrix A beside a three by four matrix B. One row of A and one column of B are highlighted in green, the colour this notebook uses for the summed index k, and their three products add to 13. The two by four product is then shown carrying only an i arrow and a j arrow. The last frame stacks two of each matrix into batches and shows the batched einsum bik,bkj to bij producing a two by two by four result." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:11px">🇪🇸 ESPAÑOL</div>La multiplicación de matrices es una contracción, y nada más. Una fila se encuentra con una columna, se suman los tres productos, y <code>k</code> —el índice que no aparece después de la flecha— desaparece. El último fotograma es el que conviene llevarse: pon <code>b</code> en ambos operandos <b>y</b> después de la flecha y se transporta en vez de sumarse, que es como una biblioteca multiplica un lote entero de matrices sin ningún bucle.</div>

In [ ]:
#@title ⏸️ Step through the animations / Recorre las animaciones { display-mode: 'form' }

# Plumbing, not a lesson. The animations above loop and then stop, and a
# GIF cannot be paused — so this fetches the same frames and hands them over
# one at a time, at whatever pace you read at.
# Plomería, no una lección: trae los mismos fotogramas y los entrega de uno en
# uno, al ritmo al que leas.

import io
import urllib.request

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

gif_urls = [
    "https://project-delphi.github.io/tensors-workshop/images/cube-06-contract.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-06-outer.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-06-matmul.gif",
]


def gif_frames(url):
    """Every frame of an animated GIF, as PNG bytes."""
    with urllib.request.urlopen(url, timeout=30) as response:
        gif = Image.open(io.BytesIO(response.read()))
    out = []
    try:
        while True:
            buffer = io.BytesIO()
            gif.convert("RGB").save(buffer, format="PNG")
            out.append(buffer.getvalue())
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    return out


try:
    gif_cache = {url: gif_frames(url) for url in gif_urls}
except Exception as error:  # offline, or the site is down
    print("EN: could not reach the site, so there are no frames to step "
          "through.", error)
    print("ES: no se pudo acceder al sitio, así que no hay fotogramas que "
          "recorrer.", error)
else:
    gif_pick = widgets.Dropdown(
        options=[(url.rsplit("/", 1)[1], url) for url in gif_urls],
        description="Animation / Animación:",
        style={"description_width": "180px"},
    )
    gif_step = widgets.IntSlider(
        min=1, max=len(gif_cache[gif_urls[0]]), value=1,
        description="Frame / Fotograma:",
        style={"description_width": "180px"},
        continuous_update=False,
    )
    gif_prev = widgets.Button(description="◀ Prev")
    gif_next = widgets.Button(description="Next ▶")
    # An Image widget, deliberately, and never widgets.Output: a payload
    # leaving an Output widget makes nbclient wait out the whole cell timeout
    # (see scripts/test_notebooks.py). This one is a plain bytes trait.
    gif_view = widgets.Image(format="png",
                             layout=widgets.Layout(max_width="100%"))

    def gif_show(*_):
        frames = gif_cache[gif_pick.value]
        gif_step.max = len(frames)
        gif_view.value = frames[min(gif_step.value, len(frames)) - 1]

    def gif_bump(delta):
        def click(_):
            frames = gif_cache[gif_pick.value]
            gif_step.value = (gif_step.value - 1 + delta) % len(frames) + 1
        return click

    gif_prev.on_click(gif_bump(-1))
    gif_next.on_click(gif_bump(+1))
    gif_pick.observe(gif_show, names="value")
    gif_step.observe(gif_show, names="value")
    gif_show()

    display(widgets.VBox([
        gif_pick,
        widgets.HBox([gif_prev, gif_step, gif_next]),
        gif_view,
    ]))


### Interactive einsum translator / Traductor interactivo de einsum

Choose an expression. The notebook will tell you:

- which indices enter;
- which index disappears;
- which indices survive;
- what the operation means.

> 🇪🇸 Elige una expresión. El cuaderno te dirá qué índices entran, cuál desaparece, cuáles sobreviven y qué significa la operación.

In [ ]:
#@title 🔤 Index-rule explorer / Explorador de reglas de índices — run me / ejecútame { display-mode: 'form' }

einsum_choice = widgets.Dropdown(
    options=[
        ("hwc,c->hw · RGB to gray / RGB a gris", "gray"),
        ("ii-> · Trace / Traza", "trace"),
        ("ij->ji · Transpose / Transpuesta", "transpose"),
        ("ik,kj->ij · Matrix product / Producto matricial", "matmul"),
        ("id,jd->ij · Similarity / Similitud", "similarity"),
    ],
    value="gray",
    description="Expression / Expresión:",
    style={"description_width": "150px"},
)

def explain_einsum_rule(choice):
    explanations = {
        "gray": (
            "hwc,c->hw",
            "c",
            "h,w",
            "multiply RGB values by RGB weights, then sum colour",
            "multiplicar valores RGB por pesos RGB y después sumar color",
        ),
        "trace": (
            "ii->",
            "i",
            "none / ninguno",
            "sum the diagonal to one scalar",
            "sumar la diagonal y obtener un escalar",
        ),
        "transpose": (
            "ij->ji",
            "none / ninguno",
            "j,i",
            "reorder axes; nothing is summed",
            "reordenar ejes; no se suma ningún índice",
        ),
        "matmul": (
            "ik,kj->ij",
            "k",
            "i,j",
            "multiply along the shared k dimension and sum it",
            "multiplicar a lo largo de la dimensión compartida k y sumarla",
        ),
        "similarity": (
            "id,jd->ij",
            "d",
            "i,j",
            "combine d features into one score for each image pair",
            "combinar d características en una puntuación para cada par de imágenes",
        ),
    }

    expr, disappears, survives, en, es = explanations[choice]

    print("Expression / Expresión:", expr)
    print("Disappears / Desaparece:", disappears)
    print("Survives / Permanece:", survives)
    print("EN:", en)
    print("ES:", es)

einsum_rule_output = widgets.interactive_output(
    explain_einsum_rule,
    {"choice": einsum_choice},
)

display(widgets.VBox([einsum_choice, einsum_rule_output]))

### Interactive RGB contraction / Contracción RGB interactiva

Move the red, green, and blue weights.

The real pixels do **not** change. Only the rule for combining the three channels changes.

The bar chart uses the actual RGB colours so you can see which channel is receiving more weight.

> 🇪🇸 Mueve los pesos rojo, verde y azul.
>
> Los píxeles reales **no cambian**. Solo cambia la regla que combina los tres canales.
>
> El gráfico de barras usa los colores RGB reales para mostrar qué canal recibe mayor peso.

In [ ]:
#@title 🎨 Colour contraction / Contracción de color — run me / ejecútame { display-mode: 'form' }

r_slider = widgets.FloatSlider(
    value=float(w[0]),
    min=0.0,
    max=1.0,
    step=0.025,
    description="R / Rojo:",
    readout_format=".3f",
    continuous_update=False,
    style={"description_width": "90px"},
)

g_slider = widgets.FloatSlider(
    value=float(w[1]),
    min=0.0,
    max=1.0,
    step=0.025,
    description="G / Verde:",
    readout_format=".3f",
    continuous_update=False,
    style={"description_width": "90px"},
)

b_slider = widgets.FloatSlider(
    value=float(w[2]),
    min=0.0,
    max=1.0,
    step=0.025,
    description="B / Azul:",
    readout_format=".3f",
    continuous_update=False,
    style={"description_width": "90px"},
)

def explore_colour_contraction(r, g, b):
    weights = np.array([r, g, b], dtype=float)
    live_gray = np.einsum("hwc,c->hw", photo, weights)

    plt.close("all")
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))

    axes[0].imshow(photo / 255.0)
    axes[0].set_title("Same real RGB input / Misma entrada RGB")
    axes[0].axis("off")

    axes[1].imshow(live_gray, cmap="gray")
    axes[1].set_title("Contracted image / Imagen contraída")
    axes[1].axis("off")

    axes[2].bar(
        ["R", "G", "B"],
        weights,
        color=["red", "green", "blue"],
    )
    axes[2].set_ylim(0, 1)
    axes[2].set_title("Weights / Pesos")
    axes[2].set_ylabel("weight / peso")

    plt.tight_layout()
    plt.show()

    print("einsum: hwc,c->hw")
    print(f"Weight sum / Suma de pesos: {weights.sum():.3f}")
    print("EN: c still disappears; only the numeric contribution of R/G/B changed.")
    print("ES: c sigue desapareciendo; solo cambió la contribución numérica de R/G/B.")

rgb_output = widgets.interactive_output(
    explore_colour_contraction,
    {"r": r_slider, "g": g_slider, "b": b_slider},
)

display(
    widgets.VBox([
        widgets.HBox([r_slider, g_slider, b_slider]),
        rgb_output,
    ])
)

## Exercise 2 — three matrix operations from real digit pixels / Ejercicio 2 — tres operaciones matriciales con píxeles reales

Matrices `A` and `B` are **not invented toy values**.

They are `2×2` central patches cut from two real `8×8` handwritten digits.

Pixel intensities in this dataset range from `0` to `16`.

We keep the matrices tiny so you can inspect the arithmetic while still using observed data.

### Three expressions / Tres expresiones

**Trace / Traza**

`ii->`

`i` disappears → diagonal values are summed.

**Transpose / Transpuesta**

`ij->ji`

No index disappears → axes are only reordered.

**Matrix product / Producto matricial**

`ik,kj->ij`

`k` disappears → multiply and sum over the shared dimension.

> 🇪🇸 `A` y `B` son recortes centrales `2×2` de dos dígitos manuscritos reales.
>
> La traza suma la diagonal, la transpuesta reordena índices y el producto matricial contrae la dimensión compartida `k`.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# Rewrite each operation with einsum and verify it against NumPy:
#
# a) trace of A           -> scalar
# b) transpose of A       -> (2, 2)
# c) matrix product A @ B -> (2, 2)
#
# ES:
# Reescribe cada operación con einsum y verifícala con NumPy:
#
# a) traza de A           -> escalar
# b) transpuesta de A     -> (2, 2)
# c) producto A @ B       -> (2, 2)
#
# For each one / Para cada una:
# - which index disappears? / ¿qué índice desaparece?
# - which index survives? / ¿qué índice permanece?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

trace_e = np.einsum("ii->", A)
transpose_e = np.einsum("ij->ji", A)
product_e = np.einsum("ik,kj->ij", A, B)

print("A — real digit label / etiqueta real:", int(digits.target[0]))
print(A)
print()
print("B — real digit label / etiqueta real:", int(digits.target[1]))
print(B)
print()

print("Trace / Traza:", trace_e, "| NumPy:", np.trace(A))
print("Transpose / Transpuesta:")
print(transpose_e)
print("Matrix product / Producto matricial:")
print(product_e)

assert np.allclose(trace_e, np.trace(A))
assert np.allclose(transpose_e, A.T)
assert np.allclose(product_e, A @ B)

print()
print("EN: all three einsum results agree with NumPy.")
print("ES: los tres resultados de einsum coinciden con NumPy.")

fig, axes = plt.subplots(1, 2, figsize=(5.5, 2.8))

for ax, idx, name in zip(
    axes,
    [0, 1],
    ["A", "B"],
):
    ax.imshow(
        digit_images[idx],
        cmap="gray_r",
        interpolation="nearest",
        vmin=0,
        vmax=16,
    )
    ax.add_patch(
        plt.Rectangle(
            (1.5, 1.5),
            2,
            2,
            fill=False,
            linewidth=2,
        )
    )
    ax.set_title(
        f"{name} · label/etiqueta {int(digits.target[idx])}\n"
        "2×2 real pixel patch / recorte real"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

### Interactive matrix-operation explorer / Explorador interactivo de operaciones matriciales

Choose an operation.

The notebook will show the result and explain the index rule in English and Spanish.

> 🇪🇸 Elige una operación. El cuaderno mostrará el resultado y explicará la regla de índices en inglés y español.

## Predict first / Predice primero

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#db2777,rgba(219,39,119,0))"></div>

Someone says the following. **Decide whether they are right before you
reveal anything** — commit to one answer, then open the check.

<div style="border-left:5px solid #db2777;background:rgba(219,39,119,0.1);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#db2777;margin-bottom:11px">THE CLAIM · LA AFIRMACIÓN</div><div style="margin:.55em 0">&ldquo;<code>ii-&gt;</code> has no index left after the arrow, so <b>it adds up every entry of the matrix</b>.&rdquo;</div></div>

A prediction you have committed to is worth more than one you keep
adjusting as the answer appears.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Alguien afirma que, como en <code>ii-&gt;</code> no queda ningún índice tras la flecha, <b>se suman todas las entradas de la matriz</b>. Decide si tiene razón <b>antes</b> de revelar la comprobación.</div>

In [ ]:
#@title 🤔 Predict: does ii-> add up the whole matrix? / Predice: ¿ii-> suma toda la matriz? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo (tested in tests/test_teaching_materials.py) ---
import numpy as np

pred_A = np.array([[5., 2.], [7., 3.]])

assert np.einsum("ii->", pred_A) == 8.0
assert pred_A.sum() == 17.0
assert np.einsum("ii->", pred_A) != pred_A.sum()
assert np.einsum("ij->", pred_A) == pred_A.sum()
# --- end counterexample / fin del contraejemplo ---

import ipywidgets as widgets
from IPython.display import display

# --- how the question is laid out / cómo se presenta la pregunta ---
# Radio buttons rather than a dropdown. Four bilingual answers squeezed into
# one 640px line were hard to read, and a dropdown hides three of them until
# you open it -- the wrong shape for a question whose whole point is weighing
# the options against each other. One per line, with room around them.
# Botones de opción en vez de un desplegable: una respuesta por línea.
import contextlib
import html as pred_html
import io

PRED_ACCENT = "#db2777"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence.

    A reading wants monospace and tight rows so the numbers line up under one
    another; a sentence wants prose type and room. The two used to share one
    13px monospace column, which is most of why the reveal read as a wall.
    """
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed.

    Exactly the same words: `check_prediction` still prints, and this catches
    what it printed and gives it typography. EN and ES stay written out as
    tags rather than becoming a colour, because a reader who cannot see the
    colour still has to be able to tell the two apart.
    """
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))

pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("Right — everything is summed / Correcto — se suma todo", "sum_all"),
        ("Wrong — repeating i selects the diagonal / Incorrecto — repetir i selecciona la diagonal", "diagonal"),
        ("Wrong — it returns the mean / Incorrecto — devuelve la media", "mean"),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)

def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de arriba\n      cuando quieras.".replace("\n      ", " "))
        return

    print("pred_A =")
    print(pred_A)
    print()
    print("einsum('ii->', pred_A) / traza:", np.einsum("ii->", pred_A))
    print("einsum('ij->', pred_A) / suma total:", np.einsum("ij->", pred_A))
    print("pred_A.sum():                     ", pred_A.sum())
    print()
    if choice == "diagonal":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: repeating the letter picks the positions where both indices agree — the diagonal — and only then sums. Two different letters, ij->, is the sum of everything. Which letters repeat decides what enters the sum.")
    print("ES: repetir la letra selecciona las posiciones donde ambos índices coinciden — la diagonal — y solo entonces suma. Con dos letras distintas, ij->, se suma todo. Qué letras se repiten decide qué entra en la suma.")

# The one <style> block in these notebooks, and the markdown rule does not
# cover it. ipywidgets gives no way to set the space between radio options
# from Python, and this is *widget output*, not a markdown cell: Colab strips
# <style> from markdown -- which is why every box in these notebooks is
# inline-styled -- but renders it in an output, the same path pandas' own
# Styler uses. Scoped to one added class so it can reach nothing else, and if
# it is ever dropped the options still work, just closer together.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

In [ ]:
#@title 🧮 Matrix operations / Operaciones matriciales — run me / ejecútame { display-mode: 'form' }

matrix_operation = widgets.ToggleButtons(
    options=[
        ("Trace / Traza", "trace"),
        ("Transpose / Transpuesta", "transpose"),
        ("Matrix product / Producto", "product"),
    ],
    value="product",
    description="Operation / Operación:",
    style={"description_width": "145px"},
)

def explore_matrix_operation(op):
    if op == "trace":
        result = np.einsum("ii->", A)
        expr = "ii->"
        en = "i is repeated and disappears, so the diagonal is summed."
        es = "i se repite y desaparece, por lo que se suma la diagonal."
        print("A =")
        print(A)

    elif op == "transpose":
        result = np.einsum("ij->ji", A)
        expr = "ij->ji"
        en = "No index disappears; i and j only exchange positions."
        es = "Ningún índice desaparece; i y j solo intercambian posiciones."
        print("A =")
        print(A)

    else:
        result = np.einsum("ik,kj->ij", A, B)
        expr = "ik,kj->ij"
        en = "k disappears; multiply along k and sum, while i and j survive."
        es = "k desaparece; se multiplica y suma sobre k, mientras i y j permanecen."
        print("A =")
        print(A)
        print("B =")
        print(B)

    print()
    print("Expression / Expresión:", expr)
    print("Result / Resultado:")
    print(result)
    print("EN:", en)
    print("ES:", es)

matrix_output = widgets.interactive_output(
    explore_matrix_operation,
    {"op": matrix_operation},
)

display(widgets.VBox([matrix_operation, matrix_output]))

### See one matrix-product cell step by step / Observa una celda del producto paso a paso

For `A @ B`, select an output position `(i,j)`.

The notebook will show exactly which products are added to create that one cell.

> 🇪🇸 Para `A @ B`, selecciona una posición de salida `(i,j)`.
>
> El cuaderno mostrará exactamente qué productos se suman para crear esa celda.

In [ ]:
#@title 🔬 One matmul cell / Una celda de matmul — run me / ejecútame { display-mode: 'form' }

i_selector = widgets.ToggleButtons(
    options=[0, 1],
    value=0,
    description="i:",
)

j_selector = widgets.ToggleButtons(
    options=[0, 1],
    value=0,
    description="j:",
)

def explain_matmul_cell(i, j):
    products = A[i, :] * B[:, j]
    value = products.sum()

    print(f"Output / Salida C[{i},{j}]")
    print()
    print(
        f"A[{i},0]×B[0,{j}] + A[{i},1]×B[1,{j}]"
    )
    print(
        f"{A[i,0]:g}×{B[0,j]:g} + "
        f"{A[i,1]:g}×{B[1,j]:g}"
    )
    print("=", float(value))
    print()
    print("EN: k takes values 0 and 1, then disappears because those products are summed.")
    print("ES: k toma los valores 0 y 1 y después desaparece porque esos productos se suman.")

matmul_cell_output = widgets.interactive_output(
    explain_matmul_cell,
    {"i": i_selector, "j": j_selector},
)

display(
    widgets.VBox([
        widgets.HBox([i_selector, j_selector]),
        matmul_cell_output,
    ])
)

## Exercise 3 — 3,229,209 similarities from real digit images / Ejercicio 3 — 3.229.209 similitudes entre imágenes reales

The handwritten digits are only `8×8` pixels.

That blocky appearance is **the original measured resolution**, not a bad download.

Each digit therefore has:

`8 × 8 = 64`

real pixel features.

After flattening:

`D.shape = (1797, 64)`

We can compare every image `i` with every image `j` using:

`id,jd->ij`

### Read the indices / Lee los índices

- `i` = query image / imagen consulta → survives / permanece
- `j` = candidate image / imagen candidata → survives / permanece
- `d` = 64 pixel features / 64 características de píxel → disappears / desaparece

So the output is:

`(1797,1797)`

one score for every image pair.

### Raw dot vs cosine / Producto punto vs coseno

Both use the same contraction.

- **Raw dot product** also depends strongly on vector magnitude or total intensity.
- **Cosine similarity** normalizes each image vector first, so it emphasizes the direction/pattern of the 64 pixel values.

> 🇪🇸 Cada dígito tiene 64 características reales de píxel.
>
> En `id,jd->ij`, `d` desaparece y se suman las 64 características; `i` y `j` permanecen.
>
> Por eso obtenemos una matriz `(1797,1797)` con una puntuación para cada par de imágenes.

In [ ]:
# TODO 4 / TAREA 4
#
# EN:
# Reshape the 1,797 real 8×8 images into:
#     D.shape == (1797, 64)
#
# ES:
# Reorganiza las 1.797 imágenes reales 8×8 para obtener:
#     D.shape == (1797, 64)
#
# TODO 5 / TAREA 5
#
# EN:
# Compute every raw dot-product similarity with ONE einsum:
#     'id,jd->ij'
#
# ES:
# Calcula todas las similitudes de producto punto con UN einsum:
#     'id,jd->ij'
#
# TODO 6 / TAREA 6
#
# EN:
# Normalize every row of D to unit length and repeat the SAME einsum
# to obtain cosine similarity.
#
# ES:
# Normaliza cada fila de D a longitud unitaria y repite el MISMO einsum
# para obtener similitud coseno.
#
# TODO 7 / TAREA 7
#
# EN:
# Use query_idx = 14.
# Exclude self-matching and compare:
# - best raw-dot match
# - top 5 cosine matches
#
# ES:
# Usa query_idx = 14.
# Excluye la coincidencia consigo misma y compara:
# - mejor coincidencia por producto punto
# - 5 mejores coincidencias por coseno
#
# Predict / Predice:
# Which index disappears in 'id,jd->ij'?
# ¿Qué índice desaparece en 'id,jd->ij'?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

D = digit_images.reshape(len(digit_images), -1)

S = np.einsum("id,jd->ij", D, D)

norms = np.linalg.norm(D, axis=1, keepdims=True)
Dn = D / np.where(norms == 0, 1.0, norms)

C = np.einsum("id,jd->ij", Dn, Dn)

assert D.shape == (1797, 64)
assert S.shape == (1797, 1797)
assert C.shape == (1797, 1797)
assert np.allclose(C.diagonal(), 1.0)

print("Flattened digits / Dígitos aplanados:", D.shape)
print("Raw similarity / Similitud producto punto:", S.shape)
print("Cosine similarity / Similitud coseno:", C.shape)
print("Pairwise scores / Puntuaciones por pares:", S.size)
print()
print("EN: d=64 pixel features disappeared; i and j survived.")
print("ES: d=64 características de píxel desapareció; i y j permanecieron.")

query_idx = 14
query_label = int(digits.target[query_idx])

raw_scores = S[query_idx].copy()
cos_scores = C[query_idx].copy()

raw_scores[query_idx] = -np.inf
cos_scores[query_idx] = -np.inf

raw_top1 = int(np.argmax(raw_scores))
cos_top5 = np.argsort(cos_scores)[-5:][::-1]

print()
print("Query / Consulta:", query_idx, "| label/etiqueta:", query_label)
print(
    "Best raw-dot match / Mejor producto punto:",
    raw_top1,
    "| label/etiqueta:",
    int(digits.target[raw_top1]),
)
print(
    "Top-5 cosine indices / Índices top-5 coseno:",
    cos_top5.tolist(),
)
print(
    "Top-5 cosine labels / Etiquetas top-5 coseno:",
    digits.target[cos_top5].astype(int).tolist(),
)

fig, axes = plt.subplots(1, 6, figsize=(11, 2.5))

axes[0].imshow(
    digit_images[query_idx],
    cmap="gray_r",
    interpolation="nearest",
    vmin=0,
    vmax=16,
)
axes[0].set_title(
    f"query / consulta\nlabel {query_label}"
)

for ax, idx in zip(axes[1:], cos_top5):
    ax.imshow(
        digit_images[idx],
        cmap="gray_r",
        interpolation="nearest",
        vmin=0,
        vmax=16,
    )
    ax.set_title(
        f"label {int(digits.target[idx])}\n"
        f"cos={C[query_idx, idx]:.3f}"
    )

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# The flattened digit matrix and the two similarity matrices (Exercise 3),
# recomputed here in a visible cell so the retrieval and pixel-vector explorers
# below run whether or not the folded solution was executed. The query-14
# ranking comparison, the assertions, and the cosine-vs-dot reasoning stay
# folded in the solution above.
D = digit_images.reshape(len(digit_images), -1)
S = np.einsum("id,jd->ij", D, D)

norms = np.linalg.norm(D, axis=1, keepdims=True)
Dn = D / np.where(norms == 0, 1.0, norms)
C = np.einsum("id,jd->ij", Dn, Dn)

### See the 64-dimensional vector / Observa el vector de 64 dimensiones

Select a digit and inspect the same data in two forms:

- the original `8×8` image;
- the flattened 64-value vector used by `id,jd->ij`.

Nothing was invented or removed — only the arrangement changed.

> 🇪🇸 Selecciona un dígito y observa los mismos datos en dos formas:
>
> - imagen original `8×8`;
> - vector aplanado de 64 valores usado por `id,jd->ij`.
>
> No se inventó ni eliminó información; solo cambió la organización.

In [ ]:
#@title 🖼️ Pixel vector / Vector de píxeles — run me / ejecútame { display-mode: 'form' }

vector_query = widgets.IntSlider(
    value=14,
    min=0,
    max=len(digit_images) - 1,
    step=1,
    description="Digit / Dígito:",
    continuous_update=False,
    style={"description_width": "95px"},
)

def show_vectorized_digit(query):
    vector = D[query]

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

    axes[0].imshow(
        digit_images[query],
        cmap="gray_r",
        interpolation="nearest",
        vmin=0,
        vmax=16,
    )
    axes[0].set_title(
        f"8×8 image / imagen\nlabel {int(digits.target[query])}"
    )
    axes[0].axis("off")

    axes[1].bar(np.arange(64), vector)
    axes[1].set_title("64 pixel features / 64 características")
    axes[1].set_xlabel("d = pixel feature / característica de píxel")
    axes[1].set_ylabel("intensity / intensidad")

    plt.tight_layout()
    plt.show()

    print("Image shape / Forma imagen:", digit_images[query].shape)
    print("Vector shape / Forma vector:", vector.shape)
    print("EN: d runs from 0 to 63 and is the index contracted in the similarity calculation.")
    print("ES: d recorre 0 a 63 y es el índice que se contrae en el cálculo de similitud.")

vector_output = widgets.interactive_output(
    show_vectorized_digit,
    {"query": vector_query},
)

display(widgets.VBox([vector_query, vector_output]))

### Interactive digit-retrieval explorer / Explorador interactivo de búsqueda de dígitos

Now turn the mathematics into a small search engine.

Choose:

- any of the 1,797 real digit images;
- **Cosine** or **Raw dot product**;
- how many neighbours to display.

The images remain intentionally pixelated because each visible square is one of the original 64 measurements.

> 🇪🇸 Ahora convierte las matemáticas en un pequeño buscador.
>
> Elige cualquier dígito real, selecciona **Coseno** o **Producto punto** y decide cuántos vecinos mostrar.
>
> Las imágenes se mantienen pixeladas intencionalmente porque cada cuadrado visible corresponde a una de las 64 mediciones originales.

In [ ]:
#@title 🔍 Retrieval explorer / Explorador de recuperación — run me / ejecútame { display-mode: 'form' }

query_slider = widgets.IntSlider(
    value=14,
    min=0,
    max=len(digit_images) - 1,
    step=1,
    description="Query / Consulta:",
    continuous_update=False,
    style={"description_width": "115px"},
)

similarity_toggle = widgets.ToggleButtons(
    options=[
        ("Cosine / Coseno", "cosine"),
        ("Raw dot / Producto punto", "raw"),
    ],
    value="cosine",
    description="Metric / Métrica:",
    style={"description_width": "100px"},
)

k_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=8,
    step=1,
    description="Top k:",
    continuous_update=False,
    style={"description_width": "55px"},
)

def explore_retrieval(query, metric, k):
    matrix = C if metric == "cosine" else S
    scores = matrix[query].copy()
    scores[query] = -np.inf

    top = np.argsort(scores)[-k:][::-1]
    q_label = int(digits.target[query])

    plt.close("all")
    fig, axes = plt.subplots(
        1,
        k + 1,
        figsize=(2.05 * (k + 1), 2.75),
    )
    axes = np.atleast_1d(axes)

    axes[0].imshow(
        digit_images[query],
        cmap="gray_r",
        interpolation="nearest",
        vmin=0,
        vmax=16,
    )
    axes[0].set_title(
        f"query {query}\nlabel {q_label}"
    )
    axes[0].axis("off")

    for ax, idx in zip(axes[1:], top):
        ax.imshow(
            digit_images[idx],
            cmap="gray_r",
            interpolation="nearest",
            vmin=0,
            vmax=16,
        )

        score_name = "cos" if metric == "cosine" else "dot"

        ax.set_title(
            f"idx {idx}\n"
            f"label {int(digits.target[idx])}\n"
            f"{score_name}={scores[idx]:.3f}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    labels = digits.target[top].astype(int).tolist()
    same_label = sum(label == q_label for label in labels)

    print("Query label / Etiqueta consulta:", q_label)
    print("Retrieved labels / Etiquetas recuperadas:", labels)
    print(
        f"Same-label matches / Coincidencias misma etiqueta: "
        f"{same_label}/{k}"
    )
    print("einsum: id,jd->ij")
    print("EN: d disappears; i and j survive.")
    print("ES: d desaparece; i y j permanecen.")

    if metric == "cosine":
        print("EN: cosine normalizes magnitude first and emphasizes pixel-pattern direction.")
        print("ES: el coseno normaliza primero la magnitud y enfatiza la dirección del patrón de píxeles.")
    else:
        print("EN: raw dot product also rewards magnitude/intensity.")
        print("ES: el producto punto también favorece magnitud/intensidad.")

retrieval_output = widgets.interactive_output(
    explore_retrieval,
    {
        "query": query_slider,
        "metric": similarity_toggle,
        "k": k_slider,
    },
)

display(
    widgets.VBox([
        query_slider,
        similarity_toggle,
        k_slider,
        retrieval_output,
    ])
)

<details>
<summary><strong>Why does cosine change the neighbours? / ¿Por qué el coseno cambia los vecinos?</strong></summary>

The raw dot product:

`D_i · D_j`

$$
\text{dot} = D_i \cdot D_j = \sum_{d=1}^{64} D_{id} D_{jd}
\qquad
\cos(D_i, D_j) = \frac{D_i \cdot D_j}{\lVert D_i\rVert \lVert D_j\rVert}
$$


gets larger when vectors have both:

- similar directions/patterns;
- large magnitudes.

Cosine similarity first divides each vector by its length.

That removes most of the magnitude effect and focuses more on the **pattern of relative pixel intensities**.

The contraction itself remains:

`id,jd->ij`

What changed was the preprocessing.

> 🇪🇸 El producto punto crudo aumenta tanto por similitud de patrón como por magnitud.
>
> La similitud coseno normaliza primero cada vector por su longitud y reduce el efecto de la magnitud.
>
> La contracción sigue siendo `id,jd->ij`; lo que cambió fue el preprocesamiento.

</details>

## What just happened / Qué acaba de pasar

You used **one index rule** at three scales.

### The sentence to remember / La frase para recordar

> **If an index disappears after `->`, it is summed. If it remains, it survives in the output.**

> 🇪🇸
>
> **Si un índice desaparece después de `->`, se suma. Si permanece, sobrevive en la salida.**

Keep this rule for section 10: a longer expression such as:

`ijk,ia,jb,kc->abc`

uses exactly the same logic.

> 🇪🇸 Conserva esta regla para la sección 10: una expresión más larga usa exactamente la misma lógica.


<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#db2777,rgba(219,39,119,0))"></div>

## Done with this section / Fin de esta sección

Next / Siguiente: **07 · Inverses and the pseudoinverse / Inversas y la pseudoinversa** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/07-inverses-and-pseudoinverse.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)